# Chuka University GraphRAG Expert System 🎓

This notebook serves as the interactive core for the Chuka University Academic Assistant. It has been synced with the production pipeline to ensure parity in retrieval accuracy and response quality.

## 1. Imports and Environment Setup
Loading dependencies and configuring environment variables. Data paths are managed relative to the project root.

In [ ]:
import os
import re
import json
import pickle
import logging
import numpy as np
import time
from dotenv import load_dotenv
from neo4j import GraphDatabase
import google.generativeai as genai

try:
    import faiss
    from sentence_transformers import SentenceTransformer
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False
    print("Warning: faiss / sentence-transformers not installed — semantic search disabled.")

# Load environment from project root
load_dotenv(os.path.join('..', '.env'))

NEO4J_URI      = os.getenv("NEO4J_URI", "")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")

GEMINI_KEYS = [
    os.getenv("GEMINI_API_KEY", ""),
    os.getenv("GEMINI_API_KEY2", ""),
    os.getenv("GEMINI_API_KEY3", "")
]
GEMINI_KEYS = [k for k in GEMINI_KEYS if k]
current_key_idx = 0

FAISS_INDEX_PATH    = os.path.join("..", "data", "faiss_index.bin")
FAISS_METADATA_PATH = os.path.join("..", "data", "faiss_metadata.pkl")

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("ChukaExpertSystem")

## 2. Robust Gemini Call Helper
Implements multi-key rotation and exponential back-off to handle free-tier rate limits.

In [ ]:
def _gemini_call(prompt: str, model_name="gemini-2.5-flash", retries=3) -> str:
    global current_key_idx
    delay = 5

    for attempt in range(retries):
        try:
            genai.configure(api_key=GEMINI_KEYS[current_key_idx])
            model = genai.GenerativeModel(model_name)
            return model.generate_content(prompt).text.strip()
        except Exception as e:
            err = str(e)
            if any(x in err.lower() for x in ["429", "quota", "rate", "503"]):
                if len(GEMINI_KEYS) > 1:
                    current_key_idx = (current_key_idx + 1) % len(GEMINI_KEYS)
                    log.warning(f"Quota reached. Using API Key #{current_key_idx + 1}")
                    continue
                if attempt < retries - 1:
                    time.sleep(delay)
                    delay *= 2
                else: return ""
            else:
                log.error(f"Gemini Error: {e}")
                return ""
    return ""

## 3. Intelligent Intent & Entity Extraction
Classifies user intent (Graph vs Semantic) and extracts academic entities with regex fallbacks.

In [ ]:
def classify_intent(query: str) -> str:
    prompt = f"""Classify this student query for Chuka University assistant routing.
Query: \"{query}\"
Rules:
- 'graph_query'      → units, past papers, timetable slots
- 'semantic_search'  → policies, fees, handbook, procedures
- 'hybrid'           → both structured and unstructured info
Return ONLY: graph_query | semantic_search | hybrid"""
    result = _gemini_call(prompt)
    for label in ["graph_query", "semantic_search", "hybrid"]: 
        if label in result: return label
    return "hybrid"

def extract_entities(query: str, profile: dict) -> dict:
    prompt = f"""Extract entities from this Chuka University query.
Query: \"{query}\"
Profile: {json.dumps(profile or {})}
Return JSON only: {{course_code, programme, year, semester, day, topic}}"""
    
    raw = _gemini_call(prompt)
    raw = re.sub(r"```(?:json)?", "", raw).strip()
    try:
        entities = json.loads(raw)
    except:
        entities = {"course_code": None, "programme": None, "year": None, "semester": None, "day": None, "topic": None}
        m_code = re.search(r'\b([A-Z]{3,5})\s*(\d{3,4})\b', query.upper())
        if m_code: entities["course_code"] = f"{m_code.group(1)} {m_code.group(2)}"
    
    if profile:
        entities["year"] = entities.get("year") or profile.get("year")
        entities["semester"] = entities.get("semester") or profile.get("semester")
        entities["programme"] = entities.get("programme") or profile.get("program")
        
    return entities

## 4. Advanced Graph & FAISS Retrieval
Full Knowledge Graph traversal logic for past papers, fees, timetables, and academic hierarchy.

In [ ]:
def retrieve_from_graph(query: str, entities: dict, profile: dict, driver) -> str:
    results = []
    code = entities.get("course_code")
    prog = entities.get("programme")
    year = entities.get("year")
    sem  = entities.get("semester")
    day  = entities.get("day")

    try:
        with driver.session() as session:
            # Past Papers
            if code:
                r = session.run("MATCH (c:CourseUnit)-[:HAS_PAST_PAPER]->(p:PastPaper) WHERE c.code =~ $pattern RETURN p.title, p.year, p.link LIMIT 5", pattern=f"(?i).*{re.escape(code)}.*")
                rows = r.data()
                if rows: results.append(f"Past papers for {code}:\n" + "\n".join([f" - [{row['p.year']}] {row['p.title']} ({row['p.link']})" for row in rows]))

            # Unit Details & Timetable
            if prog:
                cypher = "MATCH (p:Programme)-[r:HAS_UNIT]->(u:CourseUnit) WHERE p.name =~ ('(?i).*' + $prog + '.*')"
                params = {"prog": prog}
                if year: cypher += " AND r.year = $year"; params["year"] = f"Year {year}"
                if sem:  cypher += " AND r.semester = toInteger($sem)"; params["sem"] = str(sem)
                cypher += " OPTIONAL MATCH (u)-[:HAS_TIMESLOT]->(t:TimetableSlot) RETURN u.code, u.name, collect(t.raw_text) as slots LIMIT 15"
                rows = session.run(cypher, **params).data()
                if rows:
                    results.append(f"Units for {prog}:\n" + "\n".join([f" - {row['u.code']}: {row['u.name']} (Slots: {', '.join(row['slots'])})" for row in rows]))

            # Fees & Cost Calculation
            if any(kw in query.lower() for kw in ["fee", "cost", "pay"] ) and prog:
                r = session.run("MATCH (p:Programme) WHERE p.name =~ ('(?i).*' + $prog + '.*') RETURN p.name, p.fee_string, p.duration_string", prog=prog).data()
                if r:
                    p_data = r[0]
                    results.append(f"Fee structure for {p_data['p.name']}: {p_data['p.fee_string']} | Duration: {p_data['p.duration_string']}")
                    try:
                        amt = int(re.search(r'(\d+)', p_data['p.fee_string'].replace(',', '')).group(1))
                        sems = int(re.search(r'(\d+)', p_data['p.duration_string']).group(1))
                        results.append(f"Total estimated cost: KES {amt * sems:,}")
                    except: pass

    except Exception as e: log.error(f"Graph error: {e}")
    return "\n".join(results) if results else ""

def retrieve_from_faiss(query: str, index, metadata: list, embedder, k=8) -> str:
    if not FAISS_AVAILABLE or index is None: return ""
    vec = embedder.encode([query]).astype(np.float32)
    faiss.normalize_L2(vec)
    _, ids = index.search(vec, k)
    chunks = []
    for idx in ids[0]:
        if 0 <= idx < len(metadata):
            chunks.append(f"--- {metadata[idx].get('source')} (Page {metadata[idx].get('page')}) ---\n{metadata[idx].get('text')}")
    return "\n\n".join(chunks)

## 5. Synthesis & Grounding
Synthesizing the final answer based on the retrieved context, with student-friendly formatting.

In [ ]:
def synthesise_response(query: str, graph_ctx: str, faiss_ctx: str, profile: dict) -> str:
    context = ""
    if graph_ctx: context += f"[GRAPH DATA]\n{graph_ctx}\n\n"
    if faiss_ctx: context += f"[HANDBOOK DATA]\n{faiss_ctx}\n\n"
    
    prompt = f"""You are the Chuka University Virtual Assistant.
User Profile: {json.dumps(profile or {})}
Query: \"{query}\"
Context: {context or 'No specific data found.'}

Instructions:
1. Answer strictly using the context.
2. Cite handbook pages if used.
3. Format as markdown with clear bullet points.
4. If asked about fees, show the total cost calculation.
5. Be concise and helpful."""
    
    return _gemini_call(prompt)

## 6. GraphRAGAssistant Orchestrator
The production-ready class that ties all components together.

In [ ]:
class GraphRAGAssistant:
    def __init__(self):
        if not GEMINI_KEYS: raise ValueError("GEMINI_API_KEY not found in .env")
        self.driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
        self.faiss_index = None
        self.faiss_meta = []
        self.embedder = None
        if FAISS_AVAILABLE and os.path.exists(FAISS_INDEX_PATH):
            self.faiss_index = faiss.read_index(FAISS_INDEX_PATH)
            with open(FAISS_METADATA_PATH, "rb") as f: self.faiss_meta = pickle.load(f)
            self.embedder = SentenceTransformer("all-mpnet-base-v2")
            log.info("FAISS Backend Loaded.")
    
    def generate_response(self, query: str, profile: dict) -> str:
        intent = classify_intent(query)
        entities = extract_entities(query, profile)
        log.info(f"Processing: {intent} | Entities found: {list(entities.keys())}")
        
        graph_ctx = retrieve_from_graph(query, entities, profile, self.driver) if intent != 'semantic_search' else ''
        faiss_ctx = retrieve_from_faiss(query, self.faiss_index, self.faiss_meta, self.embedder) if intent != 'graph_query' else ''
        
        return synthesise_response(query, graph_ctx, faiss_ctx, profile)

    def get_mapped_programmes(self):
        with self.driver.session() as session:
            res = session.run("MATCH (p:Programme)-[:HAS_UNIT]->(u) RETURN p.name, count(u) as count ORDER BY p.name")
            return [{"name": r["p.name"], "count": r["count"]} for r in res]

    def close(self): self.driver.close()

## 7. Interactive Testing
Run the cells below to interact with the assistant.

In [ ]:
# Initialization
assistant = GraphRAGAssistant()
test_profile = {'program': 'BSc. Computer Science', 'year': 1, 'semester': 1}

In [ ]:
# Test 1: Unit & Timetable Query
print(assistant.generate_response("What units am I taking and when?", test_profile))

In [ ]:
# Test 2: Fee & Policy Query
print(assistant.generate_response("How much are the total fees for my program?", test_profile))

In [ ]:
# Cleanup
assistant.close()